# Semana 11: Práctica. El ciclo económico, la política monetaria y los mercados con datos reales

**Curso:** Tópicos de Finanzas Avanzadas (ECON-421, UPAO 2026-20)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/11_macro_mercados/clase11_practica.ipynb)

La teoría de la semana dice que la macroeconomía entra en el precio de un activo por el numerador (los flujos) y por el denominador (la tasa de descuento). Aquí lo verificamos con 25 años de datos de Estados Unidos, el país con las series más largas y mejor documentadas: medimos la inflación implícita en los bonos, clasificamos cada trimestre en una fase del ciclo, comparamos la tasa de la Reserva Federal con la regla de Taylor, vemos cómo rindieron las acciones y los bonos en cada fase y cerramos con el retorno de la Bolsa de Valores de Lima en soles y en dólares.

**Uso de este notebook.** Es material de estudio: esta semana no se desarrolla en la sesión (las horas de práctica del 18/11 son las exposiciones del informe país) y no tiene tarea asociada. Los métodos son los mismos que pide el informe país, aplicados a una muestra histórica cerrada en diciembre de 2025, de modo que los resultados no cambian con la fecha en que lo ejecutes.

**Requisito previo:** `git pull` en tu fork. Esta semana se agrega a `utils/finanzas.py` la sección "Semana 11": `fisher_real()`, `inflacion_implicita()`, `regla_taylor()`, `postura_monetaria()`, `fase_ciclo()` y `retorno_usd()`. Requiere conexión a internet (FRED, Yahoo Finance y BCRPData).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from utils import fred, bcrp
from utils.finanzas import (inflacion_implicita, regla_taylor, postura_monetaria, fase_ciclo, retorno_usd)

INICIO, FIN = "2000-01-01", "2025-12-31"          # muestra cerrada: los resultados no cambian con la fecha de ejecucion
ORDEN = ["Recuperación", "Expansión", "Desaceleración", "Contracción"]

def sombrear(ax, rec):
    """Sombrea las recesiones fechadas por el NBER (serie USREC de FRED)."""
    ax.fill_between(rec.index, 0, 1, where=rec.values > 0, transform=ax.get_xaxis_transform(), color="gray", alpha=0.25, lw=0)

## 1. Los datos: actividad, precios, empleo y tasa de política

Todo sale de FRED con `utils/fred.py`. El PBI real y el PBI potencial son trimestrales; con ellos se calcula la **brecha del producto**, $(Y - Y^*)/Y^*$, que es la medida de la teoría para ubicar la economía respecto de su capacidad. La inflación, el desempleo y la tasa de fondos federales son mensuales y se promedian por trimestre. Las franjas grises de los gráficos son las recesiones fechadas por el NBER.

In [ ]:
pbi = fred.get_series("GDPC1", "1998-01-01", FIN)                         # PBI real, trimestral
potencial = fred.get_series("GDPPOT", "1998-01-01", FIN)                  # PBI potencial (CBO), trimestral
ipc = fred.get_series("CPIAUCSL", "1998-01-01", FIN)                      # indice de precios al consumidor, mensual
subyacente = fred.get_series("PCEPILFE", "1998-01-01", FIN)               # deflactor PCE subyacente (la medida que sigue la Fed)
desempleo = fred.get_series("UNRATE", "1998-01-01", FIN)
fondos = fred.get_series("FEDFUNDS", "1998-01-01", FIN)                   # tasa efectiva de fondos federales
rec = fred.get_series("USREC", "1998-01-01", FIN)                         # 1 = recesion segun el NBER

trim = lambda s: s.resample("QS").mean()
macro = pd.DataFrame({
    "crecimiento": pbi.pct_change(4, fill_method=None) * 100,
    "brecha": (pbi / potencial - 1) * 100,
    "inflacion": trim(ipc.pct_change(12, fill_method=None) * 100),
    "subyacente": trim(subyacente.pct_change(12, fill_method=None) * 100),
    "desempleo": trim(desempleo),
    "fondos": trim(fondos)}).loc[INICIO:FIN].dropna()
print(f"Muestra: {macro.index[0]:%Y-%m} a {macro.index[-1]:%Y-%m} ({len(macro)} trimestres)\n")
print(macro.describe().loc[["mean", "min", "max"]].round(2).to_string())

fig, ax = plt.subplots(2, 2, figsize=(12, 6.5), sharex=True)
paneles = [("crecimiento", "Crecimiento del PBI real (% anual)"), ("brecha", "Brecha del producto (% del potencial)"),
           ("inflacion", "Inflacion (IPC, % 12 meses)"), ("fondos", "Tasa de fondos federales y desempleo (%)")]
for a, (col, titulo) in zip(ax.ravel(), paneles):
    a.plot(macro.index, macro[col], color="#000080", lw=1.4)
    if col == "fondos":
        a.plot(macro.index, macro["desempleo"], color="#B22222", lw=1.2, label="desempleo"); a.legend(fontsize=8)
    a.axhline(0, color="black", lw=0.6); sombrear(a, rec.loc[INICIO:FIN]); a.set_title(titulo, fontsize=10); a.grid(alpha=0.3)
plt.tight_layout()

**Lectura:** en 25 años hay tres recesiones (2001, 2008 a 2009 y 2020) y en las tres se repite la secuencia de la lámina de fases: la brecha se vuelve negativa, el desempleo sube con rapidez y la Reserva Federal recorta la tasa. Mira también lo que ocurre después de 2021: la brecha pasa a positiva, la inflación llega a su máximo de cuatro décadas y la tasa sube más de cinco puntos en año y medio. La brecha promedio de la muestra es negativa: la economía pasó más tiempo por debajo de su potencial que por encima.

## 2. Fisher en los datos: la inflación implícita

El Tesoro emite bonos nominales y bonos indexados a la inflación (TIPS) al mismo plazo. La diferencia entre sus rendimientos es la **inflación implícita**, la que el mercado necesita para que ambos bonos rindan lo mismo: $(1 + i)/(1 + r) - 1$. Es el número de la teoría (4.5 nominal y 2.0 indexado dan 2.45) calculado mes a mes desde 2003. Como han pasado más de diez años desde las primeras observaciones, podemos compararla con la inflación que efectivamente ocurrió en los diez años siguientes a cada fecha.

In [ ]:
nominal = fred.get_series("DGS10", "2003-01-01", FIN).resample("MS").mean() / 100
indexado = fred.get_series("DFII10", "2003-01-01", FIN).resample("MS").mean() / 100
bei = inflacion_implicita(nominal, indexado) * 100

# Inflacion efectivamente observada en los 10 anos siguientes a cada fecha (anualizada)
realizada = ((ipc.shift(-120) / ipc) ** (1 / 10) - 1) * 100
comp = pd.DataFrame({"implicita a 10 anos": bei, "observada en los 10 anos siguientes": realizada}).dropna()
print(f"Promedio {comp.index[0]:%Y-%m} a {comp.index[-1]:%Y-%m}:")
print(comp.mean().round(2).to_string())
print(f"\nImplicita promedio por anos: " + ", ".join(f"{a}: {v:.2f}" for a, v in bei.groupby(bei.index.year).mean().loc[[2008, 2009, 2012, 2020, 2022, 2025]].items()))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(nominal.index, nominal * 100, color="#000080", label="Tesoro nominal a 10 anos")
ax.plot(indexado.index, indexado * 100, color="#006E6E", label="Tesoro indexado (TIPS) a 10 anos")
ax.plot(bei.index, bei, color="#B22222", label="inflacion implicita")
ax.axhline(2, color="gray", ls="--", lw=0.8); sombrear(ax, rec.loc["2003":FIN])
ax.set_ylabel("%"); ax.set_title("Fisher en los datos: nominal = real + inflacion implicita"); ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()

**Lectura:** en promedio, la inflación implícita a diez años estuvo muy cerca de la inflación que luego se observó (ambas algo por encima de 2 por ciento para las fechas que ya se pueden evaluar). No es un pronóstico limpio: incluye una prima por riesgo de inflación y una prima de liquidez de los bonos indexados. Por eso cae a mínimos a fines de 2008 y en marzo de 2020, cuando nadie esperaba deflación durante diez años pero los bonos indexados perdieron liquidez. Observa además 2022: la inflación observada superó 8 por ciento y la implícita a diez años se mantuvo por debajo de 3. El mercado leyó el episodio como transitorio, y esa es la razón por la que las tasas largas subieron mucho menos que las cortas.

## 3. Las fases del ciclo, trimestre a trimestre

Aplicamos la tabla de la teoría con una regla mecánica: el **nivel** de la brecha (positiva o negativa) y su **dirección** (si sube o baja respecto de dos trimestres atrás) definen la fase.

| | Brecha sube | Brecha baja |
|---|---|---|
| **Brecha negativa** | Recuperación | Contracción |
| **Brecha positiva** | Expansión | Desaceleración |

`fase_ciclo()` devuelve la etiqueta. Luego promediamos, dentro de cada fase, el crecimiento, la inflación y los cambios del desempleo y de la tasa de política.

In [ ]:
macro["cambio_brecha"] = macro["brecha"].diff(2)                     # direccion: cambio en dos trimestres
macro["fase"] = [fase_ciclo(b, d) for b, d in zip(macro["brecha"], macro["cambio_brecha"].fillna(0))]
macro["d_fondos"] = macro["fondos"].diff(2)
macro["d_desempleo"] = macro["desempleo"].diff(2)

resumen = macro.groupby("fase").agg(trimestres=("brecha", "size"), brecha=("brecha", "mean"), crecimiento=("crecimiento", "mean"),
                                    inflacion=("inflacion", "mean"), cambio_desempleo=("d_desempleo", "mean"),
                                    cambio_fondos=("d_fondos", "mean")).reindex(ORDEN)
print(resumen.round(2).to_string())

colores = {"Recuperación": "#7FB3D5", "Expansión": "#1E8449", "Desaceleración": "#E67E22", "Contracción": "#B22222"}
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(macro.index, macro["brecha"], width=85, color=[colores[f] for f in macro["fase"]])
handles = [plt.Rectangle((0, 0), 1, 1, color=colores[f]) for f in ORDEN]
ax.axhline(0, color="black", lw=0.6); ax.set_ylabel("brecha del producto (%)"); ax.set_title("Fases del ciclo de Estados Unidos, 2000 a 2025")
ax.legend(handles, ORDEN, ncol=4, fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()

**Lectura:** compara la tabla con la lámina "Fases del ciclo económico". El crecimiento es mayor en la expansión y casi nulo en la contracción; la inflación es más alta en las dos fases de brecha positiva; el desempleo baja en la recuperación y la expansión y sube con fuerza en la contracción; la tasa de política sube en la expansión y se recorta en la contracción. La regla mecánica reproduce el patrón de la teoría, con una advertencia que el gráfico muestra bien: las fases no duran lo mismo ni se suceden siempre en orden, y un solo trimestre puede cambiar de etiqueta por una revisión del PBI. En el informe país la fase se sustenta con varios indicadores, no con uno solo.

## 4. La regla de Taylor y la postura monetaria

La regla: $i = r^* + \pi + 0.5(\pi - \pi^*) + 0.5\,\text{brecha}$. Usamos los supuestos originales de Taylor (tasa real neutral de 2 por ciento y meta de 2 por ciento) y, como medida de inflación, el deflactor subyacente del consumo, que es la que sigue la Reserva Federal. Para la **postura** comparamos la tasa real ex ante (tasa de política menos inflación esperada a un año, estimada por la Fed de Cleveland) con la tasa neutral, usando `postura_monetaria()`.

In [ ]:
R_NEUTRAL, META = 2.0, 2.0                                            # supuestos de la regla original de Taylor (1993)
macro["taylor"] = regla_taylor(R_NEUTRAL, macro["subyacente"], META, macro["brecha"])
macro["desvio"] = macro["fondos"] - macro["taylor"]

esperada = trim(fred.get_series("EXPINF1YR", "1998-01-01", FIN)).reindex(macro.index)   # inflacion esperada a un ano (Fed de Cleveland)
post = [postura_monetaria(t, e, R_NEUTRAL) for t, e in zip(macro["fondos"], esperada)]
macro["real_ex_ante"] = [p[0] for p in post]
macro["postura"] = [p[1] for p in post]

print("Tasa de fondos federales menos regla de Taylor (puntos), promedio por periodo:")
for ini, fin in (("2000", "2007"), ("2008", "2015"), ("2016", "2019"), ("2020", "2021"), ("2022", "2025")):
    print(f"  {ini} a {fin}: {macro.loc[ini:fin, 'desvio'].mean():+.2f}")
print("\nPostura por fase del ciclo (trimestres):")
print(pd.crosstab(macro["fase"], macro["postura"]).reindex(ORDEN).to_string())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(macro.index, macro["fondos"], color="#000080", label="fondos federales")
ax[0].plot(macro.index, macro["taylor"], color="#B22222", ls="--", label="regla de Taylor")
ax[0].set_title("Tasa observada y tasa de la regla (%)"); ax[0].legend(fontsize=8)
ax[1].plot(macro.index, macro["real_ex_ante"], color="#000080"); ax[1].axhline(R_NEUTRAL, color="#B22222", ls="--", lw=1)
ax[1].set_title("Tasa real ex ante y tasa neutral supuesta (%)")
for a in ax:
    sombrear(a, rec.loc[INICIO:FIN]); a.grid(alpha=0.3)
plt.tight_layout()

**Lectura:** la tasa observada estuvo casi siempre por debajo de la regla, y la distancia creció después de 2008. Hay tres explicaciones que conviene tener a mano. Primera, entre 2009 y 2015 y en 2020 a 2021 la regla pedía tasas negativas y la tasa no puede bajar de cero: el estímulo se dio con compras de activos. Segunda, la tasa neutral no es observable y la mayoría de estimaciones posteriores a 2008 la ubican por debajo de 2 por ciento: con $r^* = 0.5$ toda la línea roja baja un punto y medio. Tercera, en 2021 y 2022 la Reserva Federal tardó en reaccionar al alza de la inflación; ahí el desvío sí refleja una política rezagada. La regla es una referencia para ordenar la discusión, no una descripción de lo que hace un banco central.

La tabla cruzada cierra el argumento de la postura: en la recuperación y en la contracción la tasa real ex ante estuvo siempre por debajo de la neutral (postura expansiva), y la postura contractiva solo aparece en la expansión y en la desaceleración. Cambia `R_NEUTRAL` a 0.5 y vuelve a ejecutar la celda: ¿cuántos trimestres pasan de expansiva a neutral o contractiva? Esa sensibilidad es la razón por la que la postura se califica con un rango y no con un decimal.

## 5. El ciclo en las acciones, los bonos y el crédito

Asignamos a cada mes la fase de su trimestre y calculamos, por fase, el retorno y la volatilidad de las acciones (SPY), el retorno de los bonos del Tesoro de 7 a 10 años (IEF) y el diferencial de crédito (rendimiento de los bonos corporativos Baa menos el Tesoro a 10 años), con su cambio en seis meses.

In [ ]:
px = yf.download(["SPY", "IEF"], start="2002-07-01", end="2026-01-01", interval="1mo", auto_adjust=True, progress=False)["Close"]
ret = px.pct_change().dropna().rename(columns={"SPY": "acciones (SPY)", "IEF": "bonos del Tesoro 7 a 10 anos (IEF)"})
diferencial = fred.get_series("BAA10Y", "2002-07-01", FIN).resample("MS").mean()        # bonos Baa menos Tesoro a 10 anos

fase_m = macro["fase"].resample("MS").ffill().reindex(ret.index, method="ffill")       # cada mes hereda la fase de su trimestre
tabla = pd.DataFrame({
    "meses": ret.groupby(fase_m).size(),
    "acciones, retorno anual": ret["acciones (SPY)"].groupby(fase_m).mean() * 12 * 100,
    "acciones, volatilidad": ret["acciones (SPY)"].groupby(fase_m).std() * np.sqrt(12) * 100,
    "bonos, retorno anual": ret["bonos del Tesoro 7 a 10 anos (IEF)"].groupby(fase_m).mean() * 12 * 100,
    "diferencial de credito": diferencial.reindex(ret.index).groupby(fase_m).mean(),
    "cambio del diferencial": diferencial.diff(6).reindex(ret.index).groupby(fase_m).mean()}).reindex(ORDEN)
print(tabla.round(2).to_string())
corr = ret["acciones (SPY)"].rolling(36).corr(ret["bonos del Tesoro 7 a 10 anos (IEF)"])
print(f"\nCorrelacion acciones-bonos (36 meses): 2010 a 2019 = {corr.loc['2010':'2019'].mean():.2f} | 2022 a 2025 = {corr.loc['2022':'2025'].mean():.2f}")

**Lectura:** las acciones rinden más en la recuperación y en la expansión y casi nada en la desaceleración y la contracción, donde además su volatilidad es mayor. Es la prima por riesgo anticíclica vista desde el otro lado: los precios caen cuando la prima sube (contracción) y los mayores retornos se obtienen después, cuando la prima se comprime (recuperación). Los bonos del Tesoro hacen lo contrario: su mejor fase es la contracción, cuando la tasa de política se recorta. El diferencial de crédito es más alto en la contracción y se reduce en la recuperación, como dice la teoría.

Una cautela: estas fases se conocen después, con datos revisados. La tabla describe el ciclo; no es una estrategia de inversión, porque en tiempo real no se sabe en qué fase se está.

La última línea muestra la excepción que la teoría advierte: la correlación entre acciones y bonos fue negativa en la década de 2010 (los bonos cubrían a las acciones) y positiva desde 2022. Cuando la causa de la caída es la inflación, la tasa sube en lugar de bajar y bonos y acciones caen juntos.

## 6. Retorno en moneda local y en dólares: la Bolsa de Valores de Lima

Para un inversionista que mide su riqueza en dólares, el retorno de un índice extranjero tiene dos partes: $(1 + R_{local})(1 + \Delta\,\text{moneda}) - 1$. La variación de la moneda local frente al dólar se calcula como tipo de cambio inicial sobre tipo de cambio final, menos uno, cuando el tipo de cambio está expresado en moneda local por dólar. Lo aplicamos al Índice General de la BVL, año por año (`utils/bcrp.py`, con el CSV de `data/` como respaldo). Se usa el tipo de cambio promedio de diciembre como aproximación al de cierre.

In [ ]:
try:
    igbvl = bcrp.get_series("PN01142MM", "2014-12", "2025-12")            # Indice General de la BVL
    tc = bcrp.get_series("PN01207PM", "2014-12", "2025-12")               # soles por dolar, promedio del mes
    peru = pd.DataFrame({"igbvl": igbvl, "tc": tc})
except Exception as e:
    print("BCRPData no respondio, uso el CSV de respaldo:", e)
    peru = pd.read_csv("../data/bcrp_igbvl_tc_cobre_mensual.csv", index_col=0, parse_dates=True)[["igbvl", "tc"]].loc[:FIN]

dic = peru[peru.index.month == 12]
anual = pd.DataFrame({"retorno en soles": dic["igbvl"].pct_change(),
                      "variacion del sol": dic["tc"].shift(1) / dic["tc"] - 1}).dropna()   # sol frente al dolar: tc inicial / tc final - 1
anual["retorno en dolares"] = retorno_usd(anual["retorno en soles"], anual["variacion del sol"])
anual["suma simple"] = anual["retorno en soles"] + anual["variacion del sol"]
anual.index = anual.index.year
print((anual * 100).round(1).to_string())
acum = (1 + anual[["retorno en soles", "retorno en dolares"]]).prod() ** (1 / len(anual)) - 1
print(f"\nPromedio anual compuesto {anual.index[0]} a {anual.index[-1]}: en soles {acum.iloc[0]:.1%}, en dolares {acum.iloc[1]:.1%}")

ax = (anual[["retorno en soles", "retorno en dolares"]] * 100).plot.bar(figsize=(10, 3.8), color=["#000080", "#B22222"], width=0.75)
ax.axhline(0, color="black", lw=0.6); ax.set_ylabel("%"); ax.set_title("Indice General de la BVL: retorno anual en soles y en dolares"); ax.grid(alpha=0.3, axis="y"); plt.tight_layout()

**Lectura:** en los años en que el sol se depreció (2015, 2020 y 2021) el retorno en dólares quedó muy por debajo del retorno en soles, y en los años de apreciación ocurrió lo contrario. La columna "suma simple" muestra el error de sumar los dos componentes en lugar de componerlos: es pequeño cuando los retornos son pequeños y de varios puntos cuando son grandes (compara 2015 y 2025). Fíjate también en que los malos años de la bolsa tienden a coincidir con depreciación: para el inversionista extranjero, la moneda amplifica el riesgo del mercado local en lugar de diversificarlo. Es la misma cuenta del indicador 8 de la ficha comparativa, que cada grupo hace para los últimos 12 meses de su país.

## 7. Cierre

Desde esta semana quedan en la librería, en la sección "Semana 11" de `utils/finanzas.py`: `fisher_real()`, `inflacion_implicita()`, `regla_taylor()`, `postura_monetaria()`, `fase_ciclo()` y `retorno_usd()`.

**Para practicar por tu cuenta (sin entrega ni nota):**

1. Cambia la tasa neutral a 0.5 en la sección 4 y describe cómo cambia la lectura de la postura entre 2016 y 2019.
2. En la sección 3, mide la dirección de la brecha con cuatro trimestres en lugar de dos. ¿Qué fases cambian más?
3. Repite la sección 6 con el índice y la moneda de otro país usando `yfinance`.

**Siguiente notebook:** `12_curva_rendimientos/clase12_practica.ipynb`, la curva de rendimientos del Tesoro de Estados Unidos.